In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import  MinMaxScaler
from sklearn.linear_model import Ridge
from scipy.stats import linregress

import sys
sys.path.append('../')

from source import bag_pipeline as bag

import pickle
import matplotlib.pyplot as plt

# Create example data

In [2]:
n_samples = 1000
n_rois = 100

df = pd.DataFrame({
    "id": np.arange(n_samples),
    "age": np.random.randint(18, 85, size=n_samples),
    **{
        f"strength_{i}": np.random.randn(n_samples)
        for i in range(n_rois)
    }
})


# Save example data to CSV
df.drop("id", axis=1).to_csv("../data/example_data.csv")
df.head()

,id,age,strength_0,strength_1,strength_2,strength_3,strength_4,strength_5,strength_6,strength_7,...,strength_90,strength_91,strength_92,strength_93,strength_94,strength_95,strength_96,strength_97,strength_98,strength_99
0,0,35,-0.659696,-1.674941,0.352793,-0.095536,-1.201326,0.204624,-0.734119,0.244474,...,0.203266,-1.258245,-0.070960,0.336676,0.655005,-0.105695,0.778925,-0.336611,1.379257,-1.567943
1,1,21,-0.146513,0.050146,-0.546258,-0.167492,-0.707398,-0.529669,-2.665810,-0.394432,...,0.690346,-0.376720,-0.664953,0.349710,-2.958680,0.330477,0.138554,0.804141,0.302448,-0.334874
2,2,82,-0.842076,0.504151,0.970842,0.504315,0.708177,0.230013,-0.140345,1.936207,...,-0.697929,-0.589024,2.825532,0.751345,-0.794305,-1.391783,0.281269,-0.157196,-0.513157,1.766498
3,3,70,1.702633,0.203393,0.273914,-0.832668,0.275545,-0.399554,-0.186485,-1.762504,...,0.004993,1.398074,0.957531,1.052486,1.015134,-2.406520,1.492848,-1.031840,-0.549128,-0.316580
4,4,19,-0.390210,2.003278,1.021322,0.796327,1.321921,-1.581185,-0.709608,1.324826,...,-0.049436,0.141073,-2.820606,-2.184764,-1.397799,-0.352861,0.363042,-1.274944,1.120872,0.420096


# BAG predictions

In [3]:
# Initialize BAG pipeline parameters
# Define features and target
X = df.drop(columns=["age", "id"]).values
y = df["age"].values 

scaler = MinMaxScaler()
model = Ridge()
scoring = 'neg_mean_absolute_error'

# Define stratification quantiles to keep the age distribution in the train and test sets similar
strat_labels = pd.qcut(y, q=7, labels=False) 

# Define pipeline
pipeline = Pipeline([
                    ("scaler", scaler),
                    ("regressor", model)
                ])

RESULTS_PATH = f"../results/bag/{scaler.__class__.__name__}_{model.__class__.__name__}_{scoring}.txt"

In [4]:
# Run model predictions
y_pred, metrics, train_sum, test_sum, best_parameters, best_model = bag.nested_cv_predictions(
    X=X,
    y=y,
    strat_labels=strat_labels,
    pipeline=pipeline,
    scoring = scoring,
    )

# BAG
brain_age_gap = y_pred - y

# Age Bias Correction
slope, intercept, _, _, _ = linregress(y, brain_age_gap)
residual_gap = brain_age_gap - (slope * y + intercept)

predicted_age = y_pred
corrected_age = y + residual_gap

bag.save_performance(RESULTS_PATH, best_model, best_parameters, metrics, train_sum, test_sum, residual_gap, predicted_age, corrected_age, df)

In [5]:
# Run null model predictions
n_permutations = 1

coefs_perm = {}
chance_level = {}
for i in range(n_permutations):
    y_pred_null, metrics_null, train_sum_null, test_sum_null, best_parameters_null, best_model_null = bag.nested_cv_predictions(
        X=X,
        y=np.random.permutation(y),
        strat_labels=strat_labels,
        pipeline=pipeline,
        scoring = scoring,
        )
    
    coefs_perm[i] = best_model_null.named_steps['regressor'].coef_
    chance_level[i] = metrics_null

# Save null model results
with open(RESULTS_PATH.replace('.txt', '_null_model.pickle'), 'wb') as handle:
    pickle.dump({'coefs_perm': coefs_perm, 'chance_level': chance_level}, handle, protocol=pickle.HIGHEST_PROTOCOL)

# AgeML predictions

In [6]:
DATA_PATH = "../data/example_data.csv"

OUTPUT_FOLDER = "../results/ageml/"

scaler = "minmax"
model = "ridge"

model_name = f"{scaler}_scaler_{model}_model/"
performance_path = OUTPUT_FOLDER + model_name
           
!model_age -o $performance_path -f $DATA_PATH -m $model -s $scaler -ht 50 

/opt/homebrew/lib/python3.11/site-packages/ageml/ui.py:146: UserWarning: Directory ../results/ageml/minmax_scaler_ridge_model/ageml already exists files may be overwritten.
  warnings.warn("Directory %s already exists files may be overwritten." % self.dir_path, category=UserWarning)
Running age modelling...
/opt/homebrew/lib/python3.11/site-packages/ageml/ui.py:166: UserWarning: Directory ../results/ageml/minmax_scaler_ridge_model/ageml/model_age already exists files may be overwritten.
  warnings.warn("Directory %s already exists files may be overwritten." % self.command_dir, category=UserWarning)
-----------------------------------
Loading data...
Number of subjects in dataframe features: 1000
Removing subjects with missing data...
Removing subjects not shared among dataframes...
Final number of subjects in dataframe features: 1000 (100.00 % of initial)
No clinical information provided using all subjects as CN.
-----------------------------------
Age distribution of Controls
[Group: 